# पाठ ०५ - एजेन्टिक RAG


## सेटअप

यो नोटबुक Microsoft Agent Framework प्रयोग गरी Agentic RAG (Retrieval-Augmented Generation) ढाँचा प्रदर्शन गर्दछ।

**पूर्व आवश्यकताहरू:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — तपाईंको Azure AI Search सेवा अन्त बिन्दु
- `AZURE_SEARCH_API_KEY` — तपाईंको Azure AI Search API कुञ्जी
- वातावरण चरहरूको माध्यमबाट कन्फिगर गरिएको Azure OpenAI डिप्लोइमेन्ट
- Azure CLI प्रमाणीकृत (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## एजेन्टिक RAG के हो?

परम्परागत RAG स्थिर पाइपलाइन अनुसरण गर्छ: दस्तावेजहरू पुनःप्राप्त गर्नुहोस्, त्यसपछि प्रतिक्रिया उत्पन्न गर्नुहोस्। **एजेन्टिक RAG** अगाडि जान्छ एजेन्टलाई स्वायत्तता दिन्छ **कहिले** र **कसरी** जानकारी पुनःप्राप्त गर्ने निर्णय गर्न।

एजेन्टिक RAG सँग, एजेन्टले गर्न सक्छ:
- प्रश्नको उत्तर दिनुअघि पुनःप्राप्ति आवश्यक छ कि छैन भनेर **निर्णय गर्ने**
- कुन डेटा स्रोत वा उपकरण सोध्ने **छान्ने**
- पुनःप्राप्त परिणामहरू **मूल्यांकन गर्ने** र पहिलो प्रयास अपर्याप्त भएमा फलो-अप पुनःप्राप्तिहरू गर्ने
- बहु पुनःप्राप्ति चरणहरूबाट जानकारीलाई **समेकन गर्ने** र सुसंगत उत्तर बनाउने

यसले एजेन्टलाई स्थिर retrieve-then-generate पाइपलाइनको तुलनामा बढी लचिलो र सही बनाउँछ।


## खोज उपकरण सिर्जना गर्दै

Agentic RAG मा, बाह्य डेटा स्रोतहरूलाई **उपकरणहरू** को रूपमा प्याक गरिएको हुन्छ जुन एजेन्टले आवश्यकताअनुसार प्रयोग गर्न सक्छ। यसले एजेन्टलाई पुनःप्राप्तिलाई केवल अर्को क्रिया जस्तै व्यवहार गर्न दिन्छ, अनिवार्य कदम होइन।

तल हामीले यात्रा ज्ञान आधार परिभाषित गर्दछौं र यसलाई एउटा उपकरणको रूपमा खुलासा गर्दछौं जुन एजेन्टले गन्तव्य जानकारी खोज्न कल गर्न सक्छ।


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG एजेन्ट निर्माण

अब हामी एउटा एजेन्ट बनाउँछौं जसलाई **सधैं उत्तर दिने अघि जानकारी प्राप्त गर्न निर्देशन दिइन्छ**। एजेन्टले आफ्नो तालिम डेटा मा निर्भर हुने भन्दा ज्ञान आधारमा जवाफहरू दिन `search_travel_knowledge` उपकरण प्रयोग गर्दछ।


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## पुनरावर्ती पुनःप्राप्ति — निर्माता-जाँचकर्ता ढाँचा

Agentic RAG को एउटा मुख्य फाइदा **पुनरावर्ती पुनःप्राप्ति** हो। एजेन्टले आफ्नो प्रारम्भिक नतिजालाई पुष्टि गर्न, सुधार गर्न, वा विस्तार गर्न धेरै पटक खोज गर्न सक्छ — जसरी "निर्माता-जाँचकर्ता" कार्यप्रवाह हुन्छ:

1. **निर्माता चरण**: एजेन्टले प्रारम्भिक जानकारी प्राप्त गर्छ र प्रतिक्रिया तयार पार्छ।
2. **जाँचकर्ता चरण**: एजेन्टले विवरणहरू पुष्टि गर्न वा खाली ठाउँहरू भर्न थप पुनःप्राप्तिहरू गर्छ।

तल, एजेन्टलाई यस्तो प्रश्न सोधिएको छ जसले धेरै गन्तव्यहरूको तुलना गर्न आवश्यक पर्छ, जसले यसलाई धेरै पटक खोज्न प्रोत्साहित गर्दछ।


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## सारांश

यस पाठमा तपाईंले Microsoft Agent Framework प्रयोग गरी **Agentic RAG** प्रणाली कसरी निर्माण गर्ने भन्ने सिक्नुभयो:

- **Agentic RAG** ले एजेन्टहरूलाई स्वतन्त्र रूपमा जानकारी पुनःप्राप्ति कहिले गर्ने निर्णय गर्न दिन्छ, जसले पुनःप्राप्तिलाई निश्चित नभएर गतिशील बनाउँछ।
- **टूलहरूलाई डाटा स्रोतको रूपमा**: बाह्य ज्ञान आधारहरू (जस्तै Azure AI Search) टूलको रूपमा प्याकेज गरिन्छ जुन एजेन्टले बोलाउन सक्छ।
- **पुनरावृत्त पुनःप्राप्ति**: मेकर-चेकर ढाँचाले एजेन्टलाई धेरै पुनःप्राप्ति चरणहरू गर्ने अनुमति दिन्छ — खोज, प्रमाणिकरण, र सुधार — अन्तिम उत्तर दिनुअघि।

उत्पादनमा, तपाईंले स्मृति-आधारित `TRAVEL_KNOWLEDGE_BASE` लाई वास्तविक Azure AI Search सूचकाङ्कसँग प्रतिस्थापन गर्नुहुनेछ जसले ठूलो मात्रामा यात्रा कागजात पुनःप्राप्ति व्यवस्थापन गर्छ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यस दस्तावेजलाई AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरी अनुवाद गरिएको हो। हामी सटीकता तर्फ प्रयासरत भए तापनि, कृपया ध्यान दिनुहोस् कि स्वचालित अनुवादमा त्रुटि वा अशुद्धि हुन सक्छ। मूल दस्तावेज आफ्नो मूल भाषामै आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीको लागि व्यावसायिक मानव अनुवादको सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट हुने कुनै पनि गलतफहमी वा गलत व्याख्याको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
